# Naive Bayes: Handwritten Digits from Pixel Statistics

**Family:** Supervised learning | **Level:** Intermediate | **Dataset:** `mnist_5000` | **Runs on:** CPU, about 2 minute(s)

Classify with Bayes rule and an independence assumption, and learn why it still works.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- How Bayes rule gives a classifier
- What the naive independence assumption means
- Why Laplace smoothing is needed
- Why naive Bayes can classify well with poor probabilities

**Prerequisites:**

- Probability basics
- Bayes rule

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Naive Bayes is a classic method for text filtering and sparse count data. Binary pixels give the same structure on images.

**The problem.** Classify MNIST digits using only whether each pixel is on.

**Why it matters.** A fast, tiny model is useful as a baseline and when training data is scarce.

**Why machine learning.** We estimate per class pixel probabilities from examples.

**Why this algorithm.** Naive Bayes trains with one pass of counting. Spam detection with the SMS Spam Collection would be the textbook case, but that dataset could not be downloaded in the build environment, so images are used.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### On the Optimality of the Simple Bayesian Classifier under Zero-One Loss

- **Type:** foundational paper
- **Authors:** Domingos, P. and Pazzani, M.
- **Year and venue:** 1997, Machine Learning 29(2-3):103-130
- **DOI or URL:** https://doi.org/10.1023/A:1007413511361
- **Verification:** Crossref 2026-11-17
- **Research objective:** Explain why naive Bayes works despite the independence assumption.
- **Methodology:** Theory on zero one loss, plus experiments on many UCI datasets.
- **Dataset:** 28 UCI datasets.
- **Main findings:** Naive Bayes can be optimal even when features are dependent, because classification needs only the right ranking.
- **Limitations:** Probability estimates remain poorly calibrated.

### The Elements of Statistical Learning (2nd ed.)

- **Type:** foundational paper
- **Authors:** Hastie, T., Tibshirani, R. and Friedman, J.
- **Year and venue:** 2009, Springer
- **DOI or URL:** https://doi.org/10.1007/978-0-387-84858-7
- **Verification:** not verified in build environment
- **Research objective:** Textbook on statistical learning.
- **Methodology:** Textbook.
- **Dataset:** Not applicable.
- **Main findings:** Reference text.
- **Limitations:** Textbook.

**Relationship to this notebook.** Domingos and Pazzani explain why naive Bayes often classifies well despite dependent features. Pixels are strongly dependent, so this notebook is a direct test of that idea. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Bayes rule.** $P(c \mid x) \propto P(c)\, P(x \mid c)$.

**Naive assumption.** Features are independent given the class: $P(x \mid c) = \prod_{j=1}^p P(x_j \mid c)$.

**Bernoulli model.** For binary features, $P(x_j = 1 \mid c) = \theta_{cj}$, estimated with Laplace smoothing

$$\hat\theta_{cj} = \frac{N_{cj} + \alpha}{N_c + 2\alpha}$$

**Decision.** $\hat c = \arg\max_c \Big[\log P(c) + \sum_j x_j \log\theta_{cj} + (1-x_j)\log(1-\theta_{cj})\Big]$.

**Limitations.** Independence is false for pixels, so probabilities are over confident and poorly calibrated, but the ranking of classes can still be right.

## 5. Dataset Introduction

**MNIST subset** with 5000 images of 28 by 28 pixels (784 features, values 0 to 255) from `mlxtend.data.mnist_data`. Target `digit`, ten classes. The licence is stated as CC BY-SA 3.0 on the Hugging Face card for MNIST. Verify before reuse.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import BernoulliNB
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "naive_bayes"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("mnist_5000")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. Many pixels near the image border are always zero.

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.


In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=False, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

Each class has its own map of how likely each pixel is to be on. A new image is scored against each map and the best fitting class wins.

### 8.2 Python implementation from scratch

Bernoulli naive Bayes needs only counting and logs.

In [ ]:
ytr, yva = y_train.values, y_val.values
Btr, Bva = (Xtr > 127).astype(float), (Xva > 127).astype(float)   # pixel on or off
classes = np.unique(ytr)
prior = np.array([(ytr == c).mean() for c in classes])
alpha = 1.0
theta = np.array([(Btr[ytr == c].sum(axis=0) + alpha) / ((ytr == c).sum() + 2 * alpha) for c in classes])
log_post = Bva @ np.log(theta).T + (1 - Bva) @ np.log(1 - theta).T + np.log(prior)
mine = classes[log_post.argmax(axis=1)]
ref = BernoulliNB(alpha=1.0, binarize=127).fit(Xtr, ytr)
print("agreement with sklearn:", np.mean(mine == ref.predict(Xva)), "| validation accuracy:", np.mean(mine == yva))

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `alpha` | Laplace smoothing. Zero allows zero probabilities. |
| `binarize` | Threshold turning pixels into on or off. |
| `fit_prior` | Learn class priors. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "F1"
LOWER_IS_BETTER = False
SCORING = "f1_macro"
SWEEP_PARAM = "model__alpha"
SWEEP_VALUES = [0.01, 0.1, 0.5, 1, 5]

def build_pipeline():
    return make_model_pipeline(X_train, BernoulliNB(binarize=127), scale=False, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Classification metrics: accuracy, precision, recall, F1, ROC-AUC and PR-AUC, plus the confusion matrix. With imbalanced classes accuracy alone misleads, so the model is chosen on F1 or PR-AUC as stated in the training cell.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "BernoulliNB": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="BernoulliNB", dataset="mnist_5000",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
labels = bundle.target_names or None
fig = viz.plot_confusion(y_test, pred, labels=labels)
save_figure(fig, NOTEBOOK_ID, "confusion"); plt.show()
if y_test.nunique() == 2:
    fig = viz.plot_roc_pr(final_model, X_test, y_test)
    save_figure(fig, NOTEBOOK_ID, "roc_pr"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()
# decision regions in two PCA axes
pca_view = PCA(n_components=2, random_state=SEED).fit(Xtr)
model_for_plot = clone(final_model.named_steps["model"])
fig = viz.plot_decision_boundary(lambda: clone(model_for_plot), pca_view.transform(Xtr), y_train.values,
                                 names=["PCA 1", "PCA 2"])
save_figure(fig, NOTEBOOK_ID, "decision_regions"); plt.show()

## 12. Interpretation

Permutation importance shows which inputs the model relies on. The confusion matrix shows which classes are mixed up, and the list of misclassified rows is the starting point for error analysis. A model is only as good as the cases it fails on.

In [ ]:
print("Permutation importance is skipped: the features are raw pixels, so single pixel importance is not meaningful.")
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Class probability maps

Show the learned pixel probabilities for three digits.

In [ ]:
nb = final_model.named_steps["model"]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, digit in zip(axes, [0, 1, 8]):
    ax.imshow(np.exp(nb.feature_log_prob_[digit]).reshape(28, 28), cmap="viridis"); ax.set_title(f"P(pixel on | digit {digit})"); ax.axis("off")
plt.show()

### Experiment 2: Binarisation threshold

The threshold decides what counts as ink.

In [ ]:
from sklearn.model_selection import cross_val_score
for thr in [30, 80, 127, 180]:
    s = cross_val_score(make_model_pipeline(X_trainval, BernoulliNB(binarize=thr), scale=False), X_trainval, y_trainval, cv=5, scoring="f1_macro")
    print(f"binarize={thr}: CV macro F1 {s.mean():.3f} +- {s.std(ddof=1):.3f}")

### Experiment 3: Naive Bayes versus logistic regression

Discriminative models do not assume independence. Compare cross validated scores.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
for name, m in [("BernoulliNB", BernoulliNB(binarize=127)), ("Logistic regression", LogisticRegression(max_iter=300))]:
    s = cross_val_score(make_model_pipeline(X_trainval, m, scale=(name != "BernoulliNB")), X_trainval, y_trainval, cv=3, scoring="f1_macro")
    print(f"{name}: CV macro F1 {s.mean():.3f} +- {s.std(ddof=1):.3f}")

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Change the metric used for model choice and see whether the chosen value changes.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Domingos and Pazzani argue that naive Bayes can be optimal under zero one loss even when independence fails. The comparison with logistic regression above shows how large the practical gap is on strongly dependent pixels.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Use Gaussian or multinomial variants where they fit the features and calibrate probabilities afterwards.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- Training is counting.
- Smoothing avoids zero probabilities.
- Good ranking does not need good probabilities.

**Use cases.** Spam filtering, document classification, quick baselines.

**Limitations.** Poor probability calibration, ignores feature dependence.

**Recommended next notebook.** XGBoost.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.